# 궁·능 인력 배치 — 팀 기준 백테스트 (월 단위, 원점 12개)

**팀 합의 기준**
- 예측 원점: 2025년 매월 1일 12개, 원점마다 그달 전체(28~31일) 예측
- 학습: 2023-01-01 ~ 원점 전날 / 근무표 확정: 원점 전날
- 평가: 개소 × 날짜, 실제 휴관일(`train_exclude == 1`) 제외
- 인력 = 관람객 0.8 분위수 ÷ 2,000 (소수 그대로) / 비용 = 60만 원 × 부족 + 15만 원 × 잉여
- 예정 휴관일(`planned_closed == 1`)은 모든 방법에서 0명 배치 (권고표 규칙과 동일)

**비교하는 방법**
| 구분 | 방법 |
|---|---|
| 베이스라인 | 나이브 · 계절 나이브(7일) · 직전 7일 평균 |
| 보정 | 직전 7일 평균 + 서비스 수준 보정 (FVA 2행) |
| 후보 모형 | 개소별 배율 모형 · LightGBM 분위수(0.8) |

**실행 방법**: 런타임 → 모두 실행. 두 번째 셀에서 `train_2023_2025_core_v2.csv`(20열)를 업로드하라는 창이 뜹니다. (GPU 필요 없음)

In [ ]:
# 1. 라이브러리와 설정
import os, warnings
import numpy as np
import pandas as pd
import lightgbm as lgb
warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

DATA_FILE = "train_2023_2025_core_v2.csv"
TEST_ORIGINS = pd.date_range("2025-01-01", "2025-12-01", freq="MS")   # 백테스트 원점 12개
VIRTUAL_START = "2023-02-02"                                         # 학습용 가상 원점 시작 (매주)
MAX_H = 31                                                           # 최대 예측 거리
TAU, CU, CO, PER = 0.8, 600_000, 150_000, 2000
SEED = 42
APPLY_PLANNED_CLOSED = True     # 예정 휴관일은 0명 배치
print("lightgbm", lgb.__version__, "| pandas", pd.__version__)

In [ ]:
# 2. 데이터 불러오기 (Colab에 파일이 없으면 업로드 창이 뜹니다)
if not os.path.exists(DATA_FILE):
    from google.colab import files
    files.upload()

df = pd.read_csv(DATA_FILE, parse_dates=["date"])
df = df.sort_values(["site", "date"]).reset_index(drop=True)

# 검산: 학습 파일 기준 (25,208행, 20열, total 합 47,972,901, 실제 휴관 3,700행)
assert df.shape == (25_208, 20), df.shape
assert int(df["total"].sum()) == 47_972_901, df["total"].sum()
assert int(df["actual_closed"].sum()) == 3_700
assert df["site"].nunique() == 23
assert (df["actual_closed"] == (df["total"] == 0)).all()

# 파일에 없는 열은 있는 열에서 만듭니다 (모두 달력·휴관 규칙의 조합이라 누수 없음)
df["train_exclude"] = df["actual_closed"].astype(int)            # 실제 휴관 = 학습·평가 제외
df["day"] = df["date"].dt.day
df["is_offday"] = ((df["is_weekend"] == 1) | (df["is_holiday"] == 1)).astype(int)
df["free_open_rule"] = ((df["is_myeongjeol"] == 1) | (df["is_last_wednesday"] == 1)).astype(int)
df["free_open_type"] = np.select([df["is_myeongjeol"] == 1, df["is_last_wednesday"] == 1],
                                 ["명절연휴", "마지막수요일"], default="없음")
print("행", len(df), "| 개소", df["site"].nunique(), "| 기간", df["date"].min().date(), "~", df["date"].max().date())
print("학습·평가 가능 행 (actual_closed == 0):", int((df["train_exclude"] == 0).sum()))

In [ ]:
# 3. 날짜 단위 표시
df["is_open"] = df["train_exclude"] == 0
# 평상시 = 무료 개방일(달력 규칙)도 공휴일도 아닌 날. 달력으로 미리 아는 정보만 사용
df["is_normal"] = (df["free_open_rule"] == 0) & (df["is_holiday"] == 0)
df["date_type"] = np.select(
    [df["free_open_type"] == "명절연휴", df["free_open_type"] == "마지막수요일", df["is_holiday"] == 1, df["is_weekend"] == 1],
    ["무료개방(명절)", "무료개방(마지막 수요일)", "공휴일", "주말"], default="평일")

# 작년 같은 달·같은 요일 평균 (개방일)
ly_table = (df[df["is_open"]].groupby(["site", df["date"].dt.year.rename("yr"), "month", "dow"])["total"]
            .mean().rename("ly_same_month_dow"))
df["date_type"].value_counts()

In [ ]:
# 4. 원점 기준 피처 (모든 피처는 원점 전날까지의 개방일만 사용)
def site_origin_rows(h: pd.DataFrame, O: pd.Timestamp, H: int):
    # h: 한 개소의 전체 행 (date 오름차순). 반환: 원점 O 에서 1~H 일 뒤까지의 피처 dict 목록
    past = h[(h["date"] < O) & h["is_open"]]
    if len(past) < 28:
        return []
    win28 = past[past["date"] >= O - pd.Timedelta(days=28)]
    normal28 = win28[win28["is_normal"]]
    # 직전 28일 평상시 평균. 평상시 개방일이 5일 미만이면 최근 평상시 28개 개방일 평균으로 대체
    level = normal28["total"].mean() if len(normal28) >= 5 else past.loc[past["is_normal"], "total"].iloc[-28:].mean()
    week = past[past["date"] >= O - pd.Timedelta(days=7)]
    ma7 = week["total"].mean() if len(week) else past["total"].iloc[-7:].mean()   # 개방일 0일이면 최근 7개 개방일
    base = {"level": level, "last_open": past["total"].iloc[-1], "ma7": ma7,
            "ma7_fallback": int(len(week) == 0)}
    # 개소별 무료 개방일 배율 (원점 전 365일, 평상시 평균 대비)
    p365 = past[past["date"] >= O - pd.Timedelta(days=365)]
    nmean = p365.loc[p365["is_normal"], "total"].mean()
    fmult = {t: (p365.loc[p365["free_open_type"] == t, "total"].mean() / nmean) for t in ["명절연휴", "마지막수요일"]}
    by_dow = {}
    for d in range(7):
        s = past[past["dow"] == d]
        by_dow[d] = (s["total"].iloc[-1] if len(s) else np.nan, s["total"].iloc[-4:].mean() if len(s) else np.nan)
    rows = []
    for k in range(H):
        d = O + pd.Timedelta(days=k)
        sn, dm = by_dow[d.dayofweek]
        rows.append({"origin": O, "date": d, "horizon": k + 1, **base, "snaive7": sn, "dow_mean4": dm,
                     "fm_myeongjeol": fmult["명절연휴"], "fm_lastwed": fmult["마지막수요일"]})
    return rows


def build_origin_table(origins, horizon_of):
    out = []
    for site, h in df.groupby("site"):
        h = h.reset_index(drop=True)
        for O in origins:
            for r in site_origin_rows(h, O, horizon_of(O)):
                r["site"] = site
                out.append(r)
    t = pd.DataFrame(out)
    t = t.merge(df, on=["site", "date"], how="inner")          # 데이터 범위(~2025-12-31) 밖 날짜는 빠짐
    key = pd.MultiIndex.from_arrays([t["site"], t["date"].dt.year - 1, t["month"], t["dow"]])
    t["ly_same_month_dow"] = ly_table.reindex(key).to_numpy()  # 작년 같은 달·같은 요일 (원점보다 항상 과거)
    t["free_mult"] = np.select([t["free_open_type"] == "명절연휴", t["free_open_type"] == "마지막수요일"],
                               [t["fm_myeongjeol"], t["fm_lastwed"]], default=1.0)
    return t

virtual_origins = pd.date_range(VIRTUAL_START, "2025-12-31", freq="7D")
print("학습용 가상 원점", len(virtual_origins), "개 만드는 중...")
train_tab = build_origin_table(virtual_origins, lambda O: MAX_H)
test_tab = build_origin_table(TEST_ORIGINS, lambda O: O.days_in_month)
print("학습용", train_tab.shape, "| 백테스트용", test_tab.shape)

In [ ]:
# 5. 예측 방법들
CAT = ["site", "group", "office", "free_open_type"]
FEATS = ["horizon", "dow", "month", "day", "is_weekend", "is_holiday", "is_offday",
         "is_regular_closed_wd", "closed_wd_holiday", "closure_shifted", "planned_closed",
         "neighbor_closed_n",
         "is_last_wednesday", "seollal_offset", "chuseok_offset", "is_myeongjeol", "myeongjeol_len",
         "free_open_rule", "r_last", "r_ma7", "r_snaive", "r_dow4", "r_ly", "free_mult", "log_level"] + [c + "_c" for c in CAT]
CATS = {c: sorted(df[c].dropna().unique()) for c in CAT}

def add_model_cols(t):
    t = t.copy()
    lv = t["level"].clip(lower=1)
    for c, s in [("r_last", "last_open"), ("r_ma7", "ma7"), ("r_snaive", "snaive7"), ("r_dow4", "dow_mean4"), ("r_ly", "ly_same_month_dow")]:
        t[c] = t[s] / lv
    t["log_level"] = np.log1p(lv)
    for c in CAT:
        t[c + "_c"] = pd.Categorical(t[c], categories=CATS[c])
    return t

train_tab = add_model_cols(train_tab)
test_tab = add_model_cols(test_tab)


def predict_origin(O):
    test = test_tab[test_tab["origin"] == O].copy()
    # 정답 날짜가 원점 전날 이하인 학습 행만 (누수 방지)
    hist = train_tab[(train_tab["date"] < O) & train_tab["is_open"]]

    # 베이스라인 3개
    test["나이브"] = test["last_open"]
    test["계절 나이브"] = test["snaive7"]
    test["직전 7일 평균"] = test["ma7"]

    # 서비스 수준 보정: 원점 31일 이전에 끝난 최근 8개 가상 원점의 (실제 / 직전 7일 평균) 0.8 분위수, 개소별
    prev = hist[hist["origin"].between(O - pd.Timedelta(days=31 + 56), O - pd.Timedelta(days=31))]
    f = (prev["total"] / prev["ma7"]).groupby(prev["site"]).quantile(TAU)
    test["직전 7일 평균 + 보정"] = test["ma7"] * test["site"].map(f).fillna(1.0)

    # 개소별 배율 모형: 평상시 수준 x 배율(최근 1년, 0.8 분위수)
    h1 = hist[hist["date"] >= O - pd.Timedelta(days=365)]
    r = h1["total"] / h1["level"].clip(lower=1)
    nm = h1["is_normal"]
    m_dow = r[nm].groupby([h1.loc[nm, "site"], h1.loc[nm, "dow"]]).quantile(TAU)
    fr = h1["free_open_rule"] == 1
    m_free = r[fr].groupby([h1.loc[fr, "site"], h1.loc[fr, "free_open_type"]]).quantile(TAU)
    m_free_g = r[fr].groupby([h1.loc[fr, "group"], h1.loc[fr, "free_open_type"]]).quantile(TAU)
    hol = (~nm) & (~fr)
    m_hol = r[hol].groupby(h1.loc[hol, "group"]).quantile(TAU)
    k_dow = m_dow.reindex(pd.MultiIndex.from_frame(test[["site", "dow"]])).to_numpy()
    k_free = m_free.reindex(pd.MultiIndex.from_frame(test[["site", "free_open_type"]])).to_numpy()
    k_free_g = m_free_g.reindex(pd.MultiIndex.from_frame(test[["group", "free_open_type"]])).to_numpy()
    k_hol = test["group"].map(m_hol).to_numpy()
    mult = np.where(test["free_open_rule"] == 1, np.where(np.isnan(k_free), k_free_g, k_free),
                    np.where(test["is_holiday"] == 1, k_hol, k_dow))
    mult = np.where(np.isnan(mult), k_dow, mult)
    test["개소별 배율 모형"] = test["level"] * np.nan_to_num(mult, nan=1.0)

    # LightGBM 분위수(0.8) 글로벌 모형. 정답 = total / 평상시 수준
    model = lgb.LGBMRegressor(objective="quantile", alpha=TAU, n_estimators=500, learning_rate=0.05,
                              num_leaves=31, min_child_samples=40, subsample=0.8, subsample_freq=1,
                              colsample_bytree=0.8, random_state=SEED, n_jobs=-1, verbose=-1)
    model.fit(hist[FEATS], hist["total"] / hist["level"].clip(lower=1))
    test["LightGBM"] = np.clip(model.predict(test[FEATS]), 0, None) * test["level"].clip(lower=1).to_numpy()
    return test, model

METHODS = ["나이브", "계절 나이브", "직전 7일 평균", "직전 7일 평균 + 보정", "개소별 배율 모형", "LightGBM"]
parts, models = [], {}
for O in TEST_ORIGINS:
    p, m = predict_origin(O)
    parts.append(p); models[O] = m
    print(O.date(), "완료", len(p), "행")
res = pd.concat(parts, ignore_index=True)

if APPLY_PLANNED_CLOSED:   # 예정 휴관일은 모든 방법에서 0명
    res.loc[res["planned_closed"] == 1, METHODS] = 0.0

In [ ]:
# 6. 비용·적중률·WAPE
def cost(y, q):
    n, p = y / PER, q / PER
    return CU * np.clip(n - p, 0, None) + CO * np.clip(p - n, 0, None)

ev = res[res["is_open"]].copy()                       # 실제 휴관일 제외
for m in METHODS:
    ev["cost|" + m] = cost(ev["total"], ev[m])
    ev["short|" + m] = CU * np.clip((ev["total"] - ev[m]) / PER, 0, None)
    ev["over|" + m] = CO * np.clip((ev[m] - ev["total"]) / PER, 0, None)
    ev["hit|" + m] = ev["total"] <= ev[m]

def summary(d):
    rows = []
    for m in METHODS:
        rows.append({"방법": m, "비용(원)": d["cost|" + m].sum(), "부족 비용(원)": d["short|" + m].sum(),
                     "잉여 비용(원)": d["over|" + m].sum(), "적중률(%)": d["hit|" + m].mean() * 100,
                     "WAPE(%)": (d["total"] - d[m]).abs().sum() / d["total"].sum() * 100})
    s = pd.DataFrame(rows)
    s["직전 7일 평균 대비 절감(원)"] = s.loc[s["방법"] == "직전 7일 평균", "비용(원)"].iloc[0] - s["비용(원)"]
    return s

fmt = {"비용(원)": "{:,.0f}", "부족 비용(원)": "{:,.0f}", "잉여 비용(원)": "{:,.0f}",
       "직전 7일 평균 대비 절감(원)": "{:,.0f}", "적중률(%)": "{:.1f}", "WAPE(%)": "{:.1f}"}
print(f"평가 행 {len(ev):,} (원점 12개 x 23개소 x 그달 개방일)")
print("\n[비교표: 전체]")
s_all = summary(ev); display(s_all.style.format(fmt))
print("[비교표: 무료 개방일·공휴일을 뺀 날]")
s_norm = summary(ev[ev["is_normal"]]); display(s_norm.style.format(fmt))

In [ ]:
# 7. 절감액 분해 (요구사항 4)
c = s_all.set_index("방법")["비용(원)"]
total_save = c["직전 7일 평균"] - c["LightGBM"]
sl_save = c["직전 7일 평균"] - c["직전 7일 평균 + 보정"]
print(f"직전 7일 평균 비용      {c['직전 7일 평균']:>15,.0f} 원")
print(f"서비스 수준 보정만      {sl_save:>15,.0f} 원 절감")
print(f"모형으로 추가 절감      {total_save - sl_save:>15,.0f} 원")
print(f"전체 절감 (LightGBM)    {total_save:>15,.0f} 원 ({total_save / c['직전 7일 평균'] * 100:.1f}%)")

In [ ]:
# 8. 원점별 비용과 학습 구간 (요구사항 2: 원점별 지표, 원점별 학습 구간 날짜)
per = ev.groupby("origin")[["cost|" + m for m in METHODS]].sum().div(1e6).round(1)
per.columns = METHODS
per.insert(0, "학습 구간", [f"2023-01-01 ~ {(o - pd.Timedelta(days=1)).date()}" for o in per.index])
per.insert(1, "예측 기간", [f"{o.date()} ~ {(o + pd.offsets.MonthEnd(0)).date()}" for o in per.index])
per.index = per.index.strftime("%Y-%m")
print("[원점별 비용 (백만 원)]"); display(per)
wins = (per[METHODS].lt(per["직전 7일 평균"], axis=0)).sum()
print("직전 7일 평균보다 비용이 낮았던 원점 수 (12개 중)"); print(wins.drop("직전 7일 평균"))

In [ ]:
# 9. 유형별 비용과 적중률 (요구사항 7)
def by(col):
    g = ev.groupby(col)
    t = pd.concat({m: pd.DataFrame({"비용(백만 원)": g["cost|" + m].sum() / 1e6, "적중률(%)": g["hit|" + m].mean() * 100})
                   for m in ["직전 7일 평균", "개소별 배율 모형", "LightGBM"]}, axis=1)
    t.insert(0, ("", "행 수"), g.size())
    return t.round(1)
print("[날짜 유형별]"); display(by("date_type"))
print("[개소 유형별]"); display(by("group"))

In [ ]:
# 10. 인력이 가장 많이 모자란 10건 (LightGBM 기준) - 원인은 공지로 확인해서 채울 것
BEST = "LightGBM"
top = ev.assign(부족_인원=(ev["total"] - ev[BEST]) / PER).nlargest(10, "부족_인원")
top10 = top[["date", "site", "date_type", "total", BEST, "부족_인원", "short|" + BEST]].rename(
    columns={"total": "실제", BEST: "예측(0.8분위)", "short|" + BEST: "부족 비용(원)"})
top10["원인(공지 확인 후 작성)"] = ""
display(top10.round(1))

In [ ]:
# 11. 가이드 예시 검산: 2025-10-14(화) 창덕궁 직전 7일 평균 비용 = 1,390,175원
g = df[df["site"] == "창덕궁"]
w = g[(g["date"] >= "2025-10-07") & (g["date"] <= "2025-10-13") & g["is_open"]]
y = g.loc[g["date"] == "2025-10-14", "total"].iloc[0]
print(f"개방일 {len(w)}일 평균 {w['total'].mean():,.0f}명 / 실제 {y:,}명 / 비용 {cost(y, w['total'].mean()):,.0f}원")

In [ ]:
# 12. 결과 저장 (Colab 왼쪽 파일 탭에서 내려받을 수 있습니다)
s_all.to_csv("bt12_summary_all.csv", index=False, encoding="utf-8-sig")
s_norm.to_csv("bt12_summary_normal.csv", index=False, encoding="utf-8-sig")
per.to_csv("bt12_cost_by_origin.csv", encoding="utf-8-sig")
top10.to_csv("bt12_top10_shortage.csv", index=False, encoding="utf-8-sig")
res.to_csv("bt12_predictions.csv", index=False, encoding="utf-8-sig", date_format="%Y-%m-%d")
print("저장 완료: bt12_*.csv")